# Data Cleaning & Visualization Project

**Objective:** Clean a raw e-commerce dataset, handle missing values, duplicates and outliers, engineer useful features, visualize business insights, and tell a clear data story.

**Tools:** Python, Pandas and Matplotlib.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path.cwd()
df = pd.read_csv(BASE / "raw_ecommerce_sales.csv")
df.head()

## 1. Inspect the raw data

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.isna().sum().to_frame("Missing Values"))
print("Duplicate rows:", df.duplicated().sum())
df.describe(include="all").T

## 2. Remove duplicates and standardize text

In [ ]:
df = df.drop_duplicates().copy()

df["City"] = df["City"].astype("string").str.strip().str.title()
df["City"] = df["City"].replace({"New Delhi": "Delhi", "Bangalore": "Bengaluru"})

df["Category"] = df["Category"].astype("string").str.strip().str.title()
df["Category"] = df["Category"].replace({"Electronic": "Electronics", "Clothes": "Clothing"})

df["Payment_Method"] = df["Payment_Method"].astype("string").str.strip().str.title()
df["Payment_Method"] = df["Payment_Method"].replace({"Upi": "UPI"})

## 3. Handle missing values

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
for column in ["Customer_Age", "Unit_Price", "Customer_Rating"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["City"] = df["City"].fillna("Unknown")
df["Customer_Age"] = df["Customer_Age"].fillna(df["Customer_Age"].median())
df["Customer_Rating"] = df["Customer_Rating"].fillna(df["Customer_Rating"].median())
df["Unit_Price"] = df.groupby("Category")["Unit_Price"].transform(
    lambda s: s.fillna(s.median())
)
df["Unit_Price"] = df["Unit_Price"].fillna(df["Unit_Price"].median())

df.isna().sum()

## 4. Treat invalid values and outliers

In [ ]:
valid_age_median = df.loc[df["Customer_Age"].between(15, 80), "Customer_Age"].median()
df.loc[~df["Customer_Age"].between(15, 80), "Customer_Age"] = valid_age_median

def cap_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return series.clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

df["Unit_Price"] = cap_outliers(df["Unit_Price"])
df["Quantity"] = cap_outliers(df["Quantity"])

## 5. Feature engineering

In [ ]:
df["Gross_Sales"] = df["Quantity"] * df["Unit_Price"]
df["Net_Sales"] = df["Gross_Sales"] * (1 - df["Discount_Percent"] / 100)
df["Order_Month"] = df["Order_Date"].dt.to_period("M").astype(str)
df["Age_Group"] = pd.cut(
    df["Customer_Age"],
    bins=[14, 20, 30, 40, 50, 80],
    labels=["15-20", "21-30", "31-40", "41-50", "51+"],
    include_lowest=True
)

df.to_csv(BASE / "cleaned_ecommerce_sales.csv", index=False)
df.head()

## 6. Visual analysis

In [ ]:
category_sales = df.groupby("Category")["Net_Sales"].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
category_sales.plot(kind="bar")
plt.title("Net Sales by Product Category")
plt.xlabel("Category")
plt.ylabel("Net Sales (₹)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
monthly_sales = df.groupby("Order_Month")["Net_Sales"].sum()

plt.figure(figsize=(10, 5))
monthly_sales.plot(marker="o")
plt.title("Monthly Net Sales Trend")
plt.xlabel("Month")
plt.ylabel("Net Sales (₹)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
city_sales = df.groupby("City")["Net_Sales"].sum().sort_values(ascending=False)

plt.figure(figsize=(9, 5))
city_sales.plot(kind="bar")
plt.title("Net Sales by City")
plt.xlabel("City")
plt.ylabel("Net Sales (₹)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. Key findings

In [ ]:
print("Total net sales:", round(df["Net_Sales"].sum(), 2))
print("Average order value:", round(df["Net_Sales"].mean(), 2))
print("Average customer rating:", round(df["Customer_Rating"].mean(), 2))
print("Top category:", category_sales.index[0])
print("Top city:", city_sales.index[0])

## Conclusion

The project demonstrates a complete entry-level data workflow:

1. Inspect raw data quality.
2. Remove duplicated records.
3. standardize inconsistent labels.
4. Impute missing values using sensible rules.
5. Treat unrealistic values and statistical outliers.
6. Engineer sales and customer features.
7. Use visualizations to communicate business findings.